# Remeshing and field transfer

After the model predicts the last frame of an epoch (frame 10e+10, on mesh e), this step builds the next mesh from that predicted phase field and moves the fields onto it. Code: `remesh/remesh_transfer.py`.

**Remesh rule** (`next_mesh`, distance d = 8 length units):

1. find the phi = 0 interface of the predicted field (crossings along the mesh edges);
2. every cell within distance 8 of the interface is refined one level; groups of four cells away from it are coarsened one level;
3. 2:1 balance: neighbouring cells (across faces and corners) differ by at most one level.

The rule uses only the interface location, so small model errors away from the interface do not change the mesh.

**Field transfer** (`transfer_state`): phi and u on the new nodes are the exact FE_Q(3) values of the old field (exact on refined or unchanged cells, interpolation on coarsened cells, as in the solver); u is reset to -delta on the boundary.

This notebook shows one example and the results on all validation cases. The true solver field stands in for the prediction here; in the final pipeline the model's phase field is used. Run through HTCondor, not on the access point.

In [ ]:
cache_dir = '../../cache'
split_path = '../data/splits.json'
out_dir = 'outputs/remesh'
example_case = 'fg_25'
example_epoch = 20
workers = 8

In [ ]:
import json
import os
import sys
from multiprocessing import Pool

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))   # P1 folder
from remesh.remesh_transfer import DISTANCE, MESH_KEYS, fill_luts, leaves, next_mesh, transfer_matrix, transfer_state
from data.amr_data import FINEST, N_EPOCHS, N_FINEST, interface_points, load_epoch, load_splits

os.makedirs(out_dir, exist_ok=True)
split = load_splits(split_path)
print('remesh distance:', DISTANCE)

## Example

One transition e -> e+1: the mesh built from the phase field at the end of epoch e, next to the solver's own mesh for epoch e+1. Red: the phi = 0 interface.

In [ ]:
def draw(ax, cells, points, title):
    rects = [Rectangle((x * FINEST, y * FINEST), s * FINEST, s * FINEST) for x, y, s in cells]
    ax.add_collection(PatchCollection(rects, facecolor='none', edgecolor='0.4', linewidth=0.2))
    ax.plot(points[:, 0], points[:, 1], '.', color='red', markersize=0.5)
    ax.set_xlim(0, 500)
    ax.set_ylim(0, 500)
    ax.set_aspect('equal')
    ax.set_title(title)


d0 = load_epoch(cache_dir, example_case, example_epoch)
d1 = load_epoch(cache_dir, example_case, example_epoch + 1)
phi = d0['phi'][10].astype(np.float64)
new = next_mesh({k: d0[k] for k in MESH_KEYS}, phi)
points = interface_points(d0['pos'].astype(np.float64), d0['edge_index'], phi)
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
draw(axes[0], leaves(new), points, f'remeshed ({len(leaves(new))} cells)')
draw(axes[1], leaves(d1), points, f'solver ({len(leaves(d1))} cells)')
fig.suptitle(f'{example_case}, epoch {example_epoch} -> {example_epoch + 1}')
fig.tight_layout()
fig.savefig(os.path.join(out_dir, 'remesh_example.png'), dpi=150)

## Results on all validation cases

For every validation case and every transition e -> e+1 (15 cases x 29 transitions):

- `coverage_start`, `coverage_end`: fraction of the true phi = 0 points at the start and at the end of the next epoch that lie in finest cells of the new mesh (1 = the interface never leaves the finest cells);
- `cells_over_solver`: cells of the new mesh / cells of the solver's mesh (the cost);
- `phi_max_error`, `u_max_error_over_delta`: largest transfer error on the new nodes, against the solver field moved exactly onto the same nodes;
- `enthalpy_change`: change of H = integral of (u - phi / 2) caused by the transfer, as a fraction of the latent heat of the solid.

In [ ]:
def finest_coverage(cells, points):
    """Fraction of the points that lie in finest cells."""
    _, sizes = fill_luts(sorted(cells))
    b = np.clip(np.floor(points / FINEST).astype(np.int64), 0, N_FINEST - 1)
    return float(np.mean(sizes[b[:, 1], b[:, 0]] == 1)) if len(points) else 1.0


def enthalpy(w, phi, u):
    """H = integral of (u - phi / 2), conserved by the solver."""
    return float((w * (u - phi / 2)).sum())


def check_transition(job):
    case, e = job
    d0, d1 = load_epoch(cache_dir, case, e), load_epoch(cache_dir, case, e + 1)
    delta = float(d0['params'][0])
    phi0, u0 = d0['phi'][10].astype(np.float64), d0['u'][10].astype(np.float64)
    new = next_mesh({k: d0[k] for k in MESH_KEYS}, phi0)
    cells = leaves(new)
    pos1 = d1['pos'].astype(np.float64)
    moved = transfer_state(d0, np.column_stack([phi0, u0]), new, delta)
    truth = transfer_matrix(d1, new['pos']) @ np.column_stack([d1['phi'][0], d1['u'][0]]).astype(np.float64)
    solid = (d0['weights'] * (1 + phi0) / 2).sum()
    return {'case': case, 'epoch': e,
            'coverage_start': finest_coverage(cells, interface_points(pos1, d1['edge_index'], d1['phi'][0])),
            'coverage_end': finest_coverage(cells, interface_points(pos1, d1['edge_index'], d1['phi'][10])),
            'cells_over_solver': len(cells) / len(leaves(d1)),
            'phi_max_error': float(np.abs(moved[:, 0] - truth[:, 0]).max()),
            'u_max_error_over_delta': float(np.abs(moved[:, 1] - truth[:, 1]).max() / delta),
            'enthalpy_change': abs(enthalpy(new['weights'], moved[:, 0], moved[:, 1])
                                   - enthalpy(d0['weights'], phi0, u0)) / (0.5 * solid)}


with Pool(workers) as pool:
    rows = pd.DataFrame(pool.map(check_transition, [(c, e) for c in split['val'] for e in range(N_EPOCHS - 1)]))
rows.to_csv(os.path.join(out_dir, 'remesh_results.csv'), index=False)
summary = {'transitions': len(rows),
           'coverage_start_min': rows['coverage_start'].min(), 'coverage_end_min': rows['coverage_end'].min(),
           'cells_over_solver_mean': rows['cells_over_solver'].mean(),
           'cells_over_solver_max': rows['cells_over_solver'].max(),
           'phi_max_error': rows['phi_max_error'].max(),
           'u_max_error_over_delta': rows['u_max_error_over_delta'].max(),
           'enthalpy_change_max': rows['enthalpy_change'].max()}
with open(os.path.join(out_dir, 'summary.json'), 'w') as fh:
    json.dump({k: float(v) for k, v in summary.items()}, fh, indent=1)
pd.Series(summary)